In [ ]:
import subprocess
import sys

python = sys.executable


def run_pip(args):
    return subprocess.check_call([python, "-m", "pip", *args])


def try_install_pair(qiskit_pkg, aer_pkg):
    run_pip(["install", "--quiet", "--upgrade", "pip", "setuptools", "wheel"])
    run_pip(["uninstall", "-y", "qiskit", "qiskit-aer", "qiskit-aer-gpu", "qiskit-terra"])
    run_pip(["install", "--quiet", "--no-cache-dir", qiskit_pkg, aer_pkg])

    check_code = "import qiskit, qiskit_aer; print(qiskit.__version__, qiskit_aer.__version__)"
    subprocess.check_call([python, "-c", check_code])


candidates = [
    ("qiskit==1.2.4", "qiskit-aer-gpu==0.15.1"),
    ("qiskit==1.2.4", "qiskit-aer==0.15.1"),
    ("qiskit==1.3.2", "qiskit-aer==0.16.0"),
]

installed = False
for qiskit_pkg, aer_pkg in candidates:
    try:
        try_install_pair(qiskit_pkg, aer_pkg)
        print(f"Installed compatible stack: {qiskit_pkg} + {aer_pkg}")
        installed = True
        break
    except Exception as exc:
        print(f"Failed {qiskit_pkg} + {aer_pkg}: {exc}")

if not installed:
    raise RuntimeError("Could not install a compatible qiskit/qiskit-aer pair in this runtime.")

print("If imports still fail, restart the kernel once and run from Cell 1 again.")

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.quantum_info import Operator, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.library import SaveStatevector
import matplotlib.pyplot as plt

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

n_samples = 2000

# -------------------------
# 1. Generate Inputs
# -------------------------

# Occupancy (integer 0-100)
occupancy = np.random.randint(0, 101, n_samples)

# Solar Power (continuous 20-120 kW)
solar_kw = np.random.uniform(20, 120, n_samples)

# Temperature (20-40 C)
temperature = np.random.uniform(20, 40, n_samples)

# Base Load (constant 20 kW)
base_load = np.full(n_samples, 20)

# Grid Backup (0-40 kW)
grid_backup = np.random.uniform(0, 40, n_samples)

# Total Supply
total_supply = solar_kw + grid_backup

# -------------------------
# 2. Demand Model
# -------------------------

# Occupancy effect (scaled)
occupancy_effect = occupancy * 0.3  # each person adds 0.3 kW approx

# Temperature effect (cooling above 25 C)
temperature_effect = np.maximum(0, temperature - 25) * 1.2

# Total Demand
demand = base_load + occupancy_effect + temperature_effect

# -------------------------
# 3. Create DataFrame
# -------------------------

data = pd.DataFrame({
    "Occupancy": occupancy,
    "Solar_kW": solar_kw,
    "Temperature_C": temperature,
    "Base_Load_kW": base_load,
    "Grid_Backup_kW": grid_backup,
    "Total_Supply_kW": total_supply,
    "Demand_kW": demand,
})

print(data.head())
print("\nDataset Shape:", data.shape)

In [ ]:
# Scenario input from generated dataset
sample_idx = 0

occupancy_arr = data["Occupancy"].to_numpy(dtype=np.float64)
solar_arr = data["Solar_kW"].to_numpy(dtype=np.float64)
temperature_arr = data["Temperature_C"].to_numpy(dtype=np.float64)
baseload_arr = data["Base_Load_kW"].to_numpy(dtype=np.float64)
total_supply_arr = data["Total_Supply_kW"].to_numpy(dtype=np.float64)

occupancy = occupancy_arr[sample_idx] / 100.0  # normalize to 0-1 for this model
solar_kw = solar_arr[sample_idx]
temperature = temperature_arr[sample_idx]
baseload_kw = baseload_arr[sample_idx]
total_supply_kw = total_supply_arr[sample_idx]

# Demand model used by the quantum optimization block
demand_kw = baseload_kw + occupancy * 15.0 + max(0.0, temperature - 25.0) * 1.5

print(f"Using dataset row: {sample_idx}")
print("Demand (kW):", demand_kw)
print("Total Supply (kW):", total_supply_kw)

In [ ]:
n_qubits = 3
n_levels = 2**n_qubits

allocation_levels = np.linspace(0, total_supply_kw, n_levels)

print("Possible allocations:", allocation_levels)

In [ ]:
def build_hamiltonian(demand, baseload, total_supply):
    energies = []

    for alloc in allocation_levels:
        penalty = 0

        if alloc < baseload:
            penalty += 500  # strong constraint

        if alloc > total_supply:
            penalty += 500

        energy = (demand - alloc)**2 + penalty
        energies.append(energy)

    h_matrix = np.diag(np.asarray(energies, dtype=np.float64))
    h_operator = SparsePauliOp.from_operator(Operator(h_matrix))
    return h_matrix, h_operator

H, H_op = build_hamiltonian(demand_kw, baseload_kw, total_supply_kw)

In [ ]:
# Trainable parameters
theta = [Parameter(f'θ{i}') for i in range(12)]

def create_variational_circuit():
    qc = QuantumCircuit(n_qubits)

    # Initial superposition
    for i in range(n_qubits):
        qc.h(i)

    idx = 0

    # Layer 1
    for i in range(n_qubits):
        qc.ry(theta[idx], i)
        idx += 1
        qc.rz(theta[idx], i)
        idx += 1

    # Entanglement ring
    qc.cx(0,1)
    qc.cx(1,2)
    qc.cx(2,0)

    # Layer 2
    for i in range(n_qubits):
        qc.ry(theta[idx], i)
        idx += 1
        qc.rz(theta[idx], i)
        idx += 1

    return qc

In [ ]:
import numpy as np

ansatz = create_variational_circuit()


def _create_best_simulator():
    gpu_options = {
        "method": "statevector",
        "device": "GPU",
        "precision": "single",
        "cuStateVec_enable": True,
    }
    cpu_options = {
        "method": "statevector",
        "device": "CPU",
        "precision": "double",
    }

    try:
        sim = AerSimulator(**gpu_options)
        probe = QuantumCircuit(1)
        probe.h(0)
        probe.append(SaveStatevector(1), [0])
        sim.run(probe).result()
        return sim, "GPU", gpu_options
    except Exception as exc:
        print(f"GPU backend unavailable ({exc}). Falling back to CPU.")
        sim = AerSimulator(**cpu_options)
        return sim, "CPU", cpu_options


simulator, runtime_device, estimator_backend_options = _create_best_simulator()
ansatz_optimized = transpile(ansatz, simulator, optimization_level=3)

estimator = AerEstimator(
    backend_options=estimator_backend_options,
    transpile_options={"optimization_level": 3},
)


def energy_expectation(params, observable):
    result = estimator.run(
        circuits=[ansatz_optimized],
        observables=[observable],
        parameter_values=[params],
    ).result()
    return float(np.real(result.values[0]))

print(f"Running estimator on: {runtime_device}")

In [ ]:
def compute_gradient(params, observable):
    shift = np.pi / 2
    n_params = len(params)

    batched_params = []
    for i in range(n_params):
        plus = params.copy()
        minus = params.copy()

        plus[i] += shift
        minus[i] -= shift

        batched_params.append(plus)
        batched_params.append(minus)

    results = estimator.run(
        circuits=[ansatz_optimized] * len(batched_params),
        observables=[observable] * len(batched_params),
        parameter_values=batched_params,
    ).result()

    values = np.asarray(np.real(results.values), dtype=np.float64)
    grad = (values[0::2] - values[1::2]) / 2.0
    return grad

In [ ]:
params = np.random.uniform(0, 2 * np.pi, len(theta)).astype(np.float64)
learning_rate = 0.1
epochs = 40

energy_history = []

for epoch in range(epochs):
    grad = compute_gradient(params, H_op)
    params -= learning_rate * grad

    energy = energy_expectation(params, H_op)
    energy_history.append(energy)

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Energy: {energy:.6f}")

In [ ]:
bound = ansatz_optimized.assign_parameters(dict(zip(theta, params)))
qc_eval = bound.copy()
qc_eval.append(SaveStatevector(n_qubits), list(range(n_qubits)))

result = simulator.run(qc_eval).result()
state = np.asarray(result.get_statevector(qc_eval), dtype=np.complex128)
probabilities = np.abs(state) ** 2

best_index = np.argmax(probabilities)
optimal_allocation = allocation_levels[best_index]

print("Optimal Allocation (kW):", optimal_allocation)

deficit_surplus = total_supply_kw - optimal_allocation
print("Surplus (+) / Deficit (-):", deficit_surplus)

In [ ]:
from pathlib import Path
import json

# -------------------------
# 1) Save trained model artifacts
# -------------------------
output_dir = Path("trained_artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

artifact_npz_path = output_dir / "quantum_energy_allocator_model.npz"
artifact_meta_path = output_dir / "quantum_energy_allocator_metadata.json"

np.savez(
    artifact_npz_path,
    params=np.asarray(params, dtype=np.float64),
    energy_history=np.asarray(energy_history, dtype=np.float64),
    allocation_levels=np.asarray(allocation_levels, dtype=np.float64),
    probabilities=np.asarray(probabilities, dtype=np.float64),
    theta_names=np.asarray([str(t) for t in theta], dtype=object),
)

metadata = {
    "runtime_device": str(runtime_device),
    "epochs": int(epochs),
    "learning_rate": float(learning_rate),
    "sample_idx": int(sample_idx),
    "optimal_allocation_kw": float(optimal_allocation),
    "demand_kw": float(demand_kw),
    "total_supply_kw": float(total_supply_kw),
    "deficit_surplus_kw": float(deficit_surplus),
    "artifact_npz": str(artifact_npz_path),
}

with open(artifact_meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model artifacts to: {output_dir.resolve()}")
print(f"- Weights/history: {artifact_npz_path}")
print(f"- Metadata: {artifact_meta_path}")

# -------------------------
# 2) Visualizations
# -------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

# A. Training convergence
axes[0].plot(energy_history, linewidth=2)
axes[0].set_title("Energy Minimization Convergence")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Energy")
axes[0].grid(alpha=0.3)

# B. Allocation probability distribution
axes[1].bar(allocation_levels, probabilities, width=max(0.2, float(total_supply_kw) / (len(allocation_levels) * 1.8)))
axes[1].set_title("Final Allocation Probability")
axes[1].set_xlabel("Allocation (kW)")
axes[1].set_ylabel("Probability")
axes[1].grid(axis="y", alpha=0.3)

# C. Demand vs Supply vs Optimal Allocation
dss_labels = ["Demand", "Total Supply", "Optimal Allocation"]
dss_values = [float(demand_kw), float(total_supply_kw), float(optimal_allocation)]
axes[2].bar(dss_labels, dss_values)
axes[2].set_title("Scenario Energy Summary")
axes[2].set_ylabel("kW")
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()